## Parallel Workflow with MAF and Microsoft Foundry

![parallel-workflow](./Assets/parallel-workflow.png)

In [ ]:
%pip install -U agent-framework-core agent-framework-foundry azure-ai-projects azure-identity python-dotenv ipython

### Setting up the Environment

In [1]:
import os
from dotenv import load_dotenv
from agent_framework import Executor, WorkflowContext, handler

load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

print("Project Endpoint: ", project_endpoint)
print("Model: ", model)

Project Endpoint:  https://udemo-103-resource.services.ai.azure.com/api/projects/udemo-103
Model:  gpt-5.2


### Defining a Function to Create a Chat Agent

In [2]:
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential
from agent_framework import Agent

# Cell 2: Define async workflow
async def create_agent(agent_name: str,
                       agent_instructions: str) -> Agent:
    
    # Create async Azure credential
    credential = DefaultAzureCredential()

    # creating the Foundry Project Client
    chat_client = FoundryChatClient(
        project_endpoint=project_endpoint,
        model=model,
        credential=credential,
    )
    
    agent = Agent(
            name=agent_name,
            instructions=agent_instructions,
            client=chat_client,
        )

    print("{} Agent created successfully!".format(agent_name))
    return agent

### Creating the Location Picker Agent

In [3]:
location_picker_agent = await create_agent(
    agent_name = "Location-Picker-Agent",
    agent_instructions = """You are a helpful assistant that helps users pick a location for their vacation."""
)

Location-Picker-Agent Agent created successfully!


### Creating the Destination Recommender Agent

In [4]:
destination_recommender_agent = await create_agent(
    agent_name = "Destination-Recommender-Agent",
    agent_instructions = """You are a travel expert that provides personalized vacation recommendations based on user preferences and locations. """
)

Destination-Recommender-Agent Agent created successfully!


### Creating the Weather Agent

In [5]:
weather_agent = await create_agent(
    agent_name = "Weather-Agent",
    agent_instructions = """You are a weather expert that provides accurate and up-to-date weather information for various locations selected"""
)

Weather-Agent Agent created successfully!


### Creating the Cuisine Suggestion Agent

In [6]:
cuisine_suggestion_agent = await create_agent(
    agent_name = "Cuisine-Suggestion-Agent",
    agent_instructions = """You are a culinary expert that suggests popular local cuisines and dining options based on the selected vacation destinations."""
)

Cuisine-Suggestion-Agent Agent created successfully!


### Creating the Itinerary Planner Agent

In [7]:
itinerary_planner_agent = await create_agent(
    agent_name = "Itinerary-Planner-Agent",
    agent_instructions = """You are an itinerary planning expert that creates detailed travel itineraries based on user preferences, selected destinations, weather conditions, and local cuisine options."""
)

Itinerary-Planner-Agent Agent created successfully!


### Creating the Location Selector Agent Executor

In [8]:
class LocationSelectorExecutor(Executor):

    @handler
    async def handle(self, user_query: str, ctx: WorkflowContext[str]) -> None:
        response = await location_picker_agent.run(user_query)

        await ctx.send_message(str(response))

### Creating the Destination Recommender Agent Executor

In [9]:


class DestinationRecommenderExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await destination_recommender_agent.run(location)

        await ctx.send_message(str(response))

### Creating the Weather Agent Executor

In [10]:
class WeatherExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await weather_agent.run(location)

        await ctx.send_message(str(response))

### Creating the Cuisine Suggestion Agent Executor

In [11]:
class CuisineSuggestionExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await cuisine_suggestion_agent.run(location)

        await ctx.send_message(str(response))

### Creating the Itinerary Planner Agent Executor

In [12]:
class ItineraryPlannerExecutor(Executor):

    @handler
    async def handle(self, results: list[str], ctx: WorkflowContext[str]) -> None:
        response = await itinerary_planner_agent.run(results)

        await ctx.yield_output(str(response))

### Building the Parallel Workflow

In [13]:
# Create the executor instances / objects
location_selector_executor = LocationSelectorExecutor(id="LocationSelector")
destination_recommender_executor = DestinationRecommenderExecutor(id="DestinationRecommender")
weather_executor = WeatherExecutor(id="Weather")
cuisine_suggestion_executor = CuisineSuggestionExecutor(id="CuisineSuggestion")
itinerary_planner_executor = ItineraryPlannerExecutor(id="ItineraryPlanner")
from agent_framework import WorkflowBuilder, WorkflowViz

# Build the workflow
workflow = (
    WorkflowBuilder(start_executor=location_selector_executor)
    .add_fan_out_edges(location_selector_executor, [destination_recommender_executor, weather_executor, cuisine_suggestion_executor])
    .add_fan_in_edges([destination_recommender_executor, weather_executor, cuisine_suggestion_executor], itinerary_planner_executor)
    .build()
)

viz = WorkflowViz(workflow)

### Generating Mermaid Diagram for Visualization

In [14]:
mermaid_content = viz.to_mermaid()

# printing mermaid content as markdown
from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{mermaid_content}\n```"))

```mermaid
flowchart TD
  LocationSelector["LocationSelector (Start)"];
  DestinationRecommender["DestinationRecommender"];
  Weather["Weather"];
  CuisineSuggestion["CuisineSuggestion"];
  ItineraryPlanner["ItineraryPlanner"];
  fan_in__ItineraryPlanner__ca5ba06d((fan-in))
  CuisineSuggestion --> fan_in__ItineraryPlanner__ca5ba06d;
  DestinationRecommender --> fan_in__ItineraryPlanner__ca5ba06d;
  Weather --> fan_in__ItineraryPlanner__ca5ba06d;
  fan_in__ItineraryPlanner__ca5ba06d --> ItineraryPlanner;
  LocationSelector --> DestinationRecommender;
  LocationSelector --> Weather;
  LocationSelector --> CuisineSuggestion;
```

### Running the Workflow and Streaming Events

In [16]:
# Run the workflow and stream events in notebook

async def main():
    prompt = "help me plan a vacation to iraq with the following details: I love historical sites, prefer warm weather, and enjoy trying local foods."

    result = await workflow.run(prompt)

    print("Workflow completed with result:")
    outputs = result.get_outputs()

    for output in outputs:
        print(output)

await main()



Workflow completed with result:
Absolutely — with just those 4 inputs I can lock this into a **day-by-day, food-forward itinerary** (what to eat *where*, best times of day for heat, and how to bundle Ur + marshes smoothly).

### 1) Passport nationality
This affects **visa-on-arrival vs e-visa**, and (sometimes) whether it’s simpler to do **federal Iraq + Kurdistan** on one entry plan.

### 2) Month + total days (and rough dates if you have them)
This determines whether we:
- front-load outdoor ruins (Babylon/Ur) at dawn,
- add Kurdistan for cooler evenings,
- shorten Basra/marsh time if humidity will be punishing.

### 3) Travel style: private driver/guide vs independent
Pick the closest:
- **A)** Mostly independent (taxis/apps, occasional day guide)  
- **B)** Mix (guided day trips + self-guided city days)  
- **C)** Full private driver/guide for intercity legs + key sites (smoothest for Samarra / southern loop)

### 4) Food preferences & boundaries
Any of these?  
- seafood yes/no (B